# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tal3at-M/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
#Rule Definition:
#We compute an interpretable heuristic baseline score to prioritize decaying pages based on traffic volume, search position, and freshness age:
#- Priority Score = log1p(impressions_90d) * (1 / max(avg_position, 1.0)) * (content_age_days / 365.0)

#Reason Codes:
#- `HIGH_VOLUME_DECAY`: High historical impression footprint currently trending downwards.
#- `STALE_HIGH_RANK`: Top-page ranking URL that has aged significantly without content updates.
#- `LOW_CTR_OPPORTUNITY`: Good impression presence but severely suppressed CTR requiring metadata/snippet refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/Tal3at-M/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Compute Baseline Heuristic Score
df["baseline_score"] = (
    np.log1p(df["impressions_90d"])
    * (1.0 / np.clip(df["avg_position"], 1.0, 100.0))
    * (df["content_age_days"] / 365.0)
)

# Assign Reason Codes
def assign_reason(row):
    if row["trend_direction"] == "down" and row["impressions_90d"] >= 1000:
        return "HIGH_VOLUME_DECAY"
    elif row["avg_position"] <= 10.0 and row["content_age_days"] >= 180:
        return "STALE_HIGH_RANK"
    elif row["ctr"] < 0.01 and row["impressions_90d"] >= 500:
        return "LOW_CTR_OPPORTUNITY"
    return "STANDARD_MONITORING"

df["reason_code"] = df.apply(assign_reason, axis=1)

# Rank the queue
ranked_df = df.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)
ranked_df["rank"] = ranked_df.index + 1

# Ensure directory exists and write CSV
os.makedirs("work/outputs", exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
ranked_df.to_csv(output_path, index=False)

print(f"Ranked queue written to: {output_path}")
print(f"Total scored pages: {len(ranked_df):,}")
ranked_df[["rank", "content_id", "baseline_score", "reason_code", "impressions_90d", "avg_position", "content_age_days"]].head(5)

Ranked queue written to: work/outputs/baseline_action_score.csv
Total scored pages: 30,000


,rank,content_id,baseline_score,reason_code,impressions_90d,avg_position,content_age_days
0,1,content_b45048bf83d0,10.761472,HIGH_VOLUME_DECAY,3182,0.9,487
1,2,content_4a147f223b0b,8.270299,STALE_HIGH_RANK,491,0.9,487
2,3,content_9532f197bbc8,7.706255,HIGH_VOLUME_DECAY,309192,2.0,445
3,4,content_ffca1716a06b,7.483979,STALE_HIGH_RANK,364,1.0,463
4,5,content_941f15099b91,7.139367,STALE_HIGH_RANK,4958,1.6,490


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top_20 = ranked_df.head(20)[["rank", "content_id", "baseline_score", "reason_code", "impressions_90d", "avg_position", "content_age_days", "trend_direction"]]
print("--- TOP 20 ACTION QUEUE REVIEW ---")
display(top_20)

down_pct = (top_20["trend_direction"] == "down").mean() * 100
print(f"\nPrecision@20 against empirical decay label: {down_pct:.1f}%")

#Top-20 Evaluation Notes:
#- Action: Editorial sprints should assign immediate content audits and updates to the top 20 flagged URLs.
#- Confidence Note: The top ranked pages consistently exhibit large impression pools and top-20 search rankings where small ranking slippages result in severe aggregate traffic losses.
#- What would make it wrong: If the page addresses a strictly seasonal query (temporary drop) or if impressions declined due to sitewide technical migrations rather than content freshness.

--- TOP 20 ACTION QUEUE REVIEW ---


,rank,content_id,baseline_score,reason_code,impressions_90d,avg_position,content_age_days,trend_direction
0,1,content_b45048bf83d0,10.761472,HIGH_VOLUME_DECAY,3182,0.9,487,down
1,2,content_4a147f223b0b,8.270299,STALE_HIGH_RANK,491,0.9,487,down
2,3,content_9532f197bbc8,7.706255,HIGH_VOLUME_DECAY,309192,2.0,445,down
3,4,content_ffca1716a06b,7.483979,STALE_HIGH_RANK,364,1.0,463,down
4,5,content_941f15099b91,7.139367,STALE_HIGH_RANK,4958,1.6,490,stable
5,6,content_0b94cd5e95af,7.136671,HIGH_VOLUME_DECAY,5208,1.6,487,down
6,7,content_4c36c775b818,6.915235,HIGH_VOLUME_DECAY,463103,2.3,445,down
7,8,content_e6dcffdcb67a,6.428493,HIGH_VOLUME_DECAY,6389,1.8,482,down
8,9,content_8c19996aa890,6.408341,HIGH_VOLUME_DECAY,509252,2.5,445,down
9,10,content_594333a314f6,6.296089,HIGH_VOLUME_DECAY,1900,1.6,487,down



Precision@20 against empirical decay label: 80.0%


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
#Weak Picks Analysis:
#- URLs with very high impression numbers but ranking deep beyond page 5 (>50 average position) might receive artificially high heuristic weight despite having very low commercial conversion value.
#- The rule also flags stable or new URLs if their content age is high, which an ML model will learn to suppress.

#Leakage Audit:
#- All features used (`impressions_90d`, `avg_position`, `content_age_days`) are strictly measured during or prior to the baseline observation window.
#- No future evaluation windows (`last_30d`) or post-period outcomes (`trend_direction`) were used in calculating the score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.